In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# 1. Carregamento dos dados processados
train_df = pd.read_csv('../dados/train_processado.csv')
test_df = pd.read_csv('../dados/test_processado.csv')

Modelo 1 (Random Forest com K-Fold - Acurácia K-Fold: 83.16% | Kaggle: 78,47%)

In [ ]:
features = ['ClassePassageiro', 'Sexo', 'Idade', 'Total_Familiar', 'ValorBilhete_Pessoa']
X = train_df[features]
y = train_df['Sobreviveu']
X_teste = test_df[features].reindex(columns=features, fill_value=0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
previsoes_totais = np.zeros(len(X_teste))
acuracias_folds = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_split=5,
        random_state=42
    )
    modelo_rf.fit(X_treino, y_treino)
    
    preds_val = modelo_rf.predict(X_val)
    acc = accuracy_score(y_val, preds_val)
    acuracias_folds.append(acc)
    print(f"Acurácia do Fold {fold + 1}: {acc * 100:.2f}%")
    
    previsoes_totais += modelo_rf.predict(X_teste) / skf.n_splits

print(f"\nAcurácia Média da Random Forest com K-Fold: {np.mean(acuracias_folds) * 100:.2f}%")
previsoes_finais = (previsoes_totais >= 0.5).astype(int)

submissao = pd.DataFrame({
    'PassengerId': test_df['Id_Passageiro'],
    'Survived': previsoes_finais
})
submissao.to_csv('../dados/resultado_submission_rf_kfolds.csv', index=False)
print("Arquivo de submissão gerado com sucesso!")

Acurácia do Fold 1: 84.36%
Acurácia do Fold 2: 81.46%
Acurácia do Fold 3: 82.02%
Acurácia do Fold 4: 83.15%
Acurácia do Fold 5: 84.83%

Acurácia Média da Random Forest com K-Fold: 83.16%
Arquivo de submissão gerado com sucesso!


Modelo 2 (Random Forest com GridSearchCV + K-Fold - Acurácia K-Fold: 82.27% | Kaggle: 75,60%)

In [ ]:
features_m2 = ['ClassePassageiro', 'Sexo', 'ValorBilhete_Pessoa', 'Titulo_Mr', 'Titulo_Mrs', 'Titulo_Miss', 'Sozinho']
X_m2 = train_df[features_m2]
X_teste_m2 = test_df[features_m2].reindex(columns=features_m2, fill_value=0)

rf_base = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=skf, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_m2, y)

print("Melhores parâmetros encontrados:", grid_search.best_params_)
print(f"Melhor acurácia na Validação Cruzada: {grid_search.best_score_ * 100:.2f}%\n")

previsoes_totais_m2 = np.zeros(len(X_teste_m2))
acuracias_folds_m2 = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X_m2, y)):
    X_treino, y_treino = X_m2.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X_m2.iloc[val_idx], y.iloc[val_idx]
    
    modelo_rf_grid = RandomForestClassifier(**grid_search.best_params_, random_state=42)
    modelo_rf_grid.fit(X_treino, y_treino)
    
    preds_val = modelo_rf_grid.predict(X_val)
    acc = accuracy_score(y_val, preds_val)
    acuracias_folds_m2.append(acc)
    previsoes_totais_m2 += modelo_rf_grid.predict(X_teste_m2) / skf.n_splits

print(f"Acurácia Média da Random Forest Otimizada: {np.mean(acuracias_folds_m2) * 100:.2f}%")
sub_m2 = pd.DataFrame({'PassengerId': test_df['Id_Passageiro'], 'Survived': (previsoes_totais_m2 >= 0.5).astype(int)})
sub_m2.to_csv('../dados/resultado_submission_rf_grid_kfolds.csv', index=False)

Melhores parâmetros encontrados: {'max_depth': 6, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Melhor acurácia na Validação Cruzada: 82.27%

Acurácia Média da Random Forest Otimizada: 82.27%


Modelo 3 (Random Forest Otimizada com Features Expandidas - Acurácia K-Fold: 83.16% | Kaggle: 77,50%)

In [ ]:
features_m3 = ['ClassePassageiro', 'Sexo', 'Idade', 'Total_Familiar', 'ValorBilhete_Pessoa', 'Titulo_Mr', 'Titulo_Mrs', 'Titulo_Miss']
X_m3 = train_df[features_m3]
X_teste_m3 = test_df[features_m3].reindex(columns=features_m3, fill_value=0)

previsoes_totais_m3 = np.zeros(len(X_teste_m3))
acuracias_folds_m3 = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X_m3, y)):
    X_treino, y_treino = X_m3.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X_m3.iloc[val_idx], y.iloc[val_idx]
    
    modelo_rf_opt = RandomForestClassifier(n_estimators=150, max_depth=5, min_samples_split=6, min_samples_leaf=2, random_state=42)
    modelo_rf_opt.fit(X_treino, y_treino)
    
    acc = accuracy_score(y_val, modelo_rf_opt.predict(X_val))
    acuracias_folds_m3.append(acc)
    previsoes_totais_m3 += modelo_rf_opt.predict(X_teste_m3) / skf.n_splits

print(f"Acurácia Média: {np.mean(acuracias_folds_m3) * 100:.2f}%")
sub_m3 = pd.DataFrame({'PassengerId': test_df['Id_Passageiro'], 'Survived': (previsoes_totais_m3 >= 0.5).astype(int)})
sub_m3.to_csv('../dados/resultado_submission_rf_otimizado.csv', index=False)

Acurácia Média: 83.16%


Modelo 4 (GridSearch Avançado + Threshold Tuning - Acurácia K-Fold: 83.84% | Kaggle: 78,47%)

In [ ]:
param_grid_m4 = {'n_estimators': [100, 150, 200], 'max_depth': [4, 5, 6, 7], 'min_samples_split': [4, 6, 8], 'min_samples_leaf': [1, 2, 4]}
grid_m4 = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid_m4, cv=skf, scoring='accuracy', n_jobs=-1)
grid_m4.fit(X, y)

oof_probs = np.zeros(len(X))
test_probs = np.zeros(len(X_teste))

for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    modelo_fold = RandomForestClassifier(**grid_m4.best_params_, random_state=42)
    modelo_fold.fit(X_treino, y_treino)
    
    oof_probs[val_idx] = modelo_fold.predict_proba(X_val)[:, 1]
    test_probs += modelo_fold.predict_proba(X_teste)[:, 1] / skf.n_splits

melhor_limiar = 0.5
melhor_acc_limiar = 0
for limiar in np.arange(0.30, 0.70, 0.01):
    acc_temp = accuracy_score(y, (oof_probs >= limiar).astype(int))
    if acc_temp > melhor_acc_limiar:
        melhor_acc_limiar = acc_temp
        melhor_limiar = limiar

print(f"Melhor limiar: {melhor_limiar:.2f} | Acurácia Otimizada: {melhor_acc_limiar * 100:.2f}%")
sub_m4 = pd.DataFrame({'PassengerId': test_df['Id_Passageiro'], 'Survived': (test_probs >= melhor_limiar).astype(int)})
sub_m4.to_csv('../dados/resultado_submission_otimizado_threshold.csv', index=False)

Melhor limiar: 0.50 | Acurácia Otimizada: 83.84%


Modelo 5 (XGBoost Padrão - Acurácia K-Fold: 83.50% | Kaggle: 75,12%)

In [ ]:
previsoes_totais_xgb = np.zeros(len(X_teste))
acuracias_xgb = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    modelo_xgb = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
    modelo_xgb.fit(X_treino, y_treino)
    
    acc = accuracy_score(y_val, modelo_xgb.predict(X_val))
    acuracias_xgb.append(acc)
    previsoes_totais_xgb += modelo_xgb.predict_proba(X_teste)[:, 1] / skf.n_splits

print(f"Acurácia Média XGBoost: {np.mean(acuracias_xgb) * 100:.2f}%")
sub_xgb = pd.DataFrame({'PassengerId': test_df['Id_Passageiro'], 'Survived': (previsoes_totais_xgb >= 0.5).astype(int)})
sub_xgb.to_csv('../dados/resultado_submission_xgboost.csv', index=False)

Acurácia Média XGBoost: 83.50%


Modelo 6 (XGBoost Regularizado - Acurácia K-Fold: 82.49% | Kaggle: 77,99%)

In [ ]:
previsoes_totais_xgbr = np.zeros(len(X_teste))
acuracias_xgbr = []

for fold, (treino_idx, val_idx) in enumerate(skf.split(X, y)):
    X_treino, y_treino = X.iloc[treino_idx], y.iloc[treino_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    modelo_xgbr = xgb.XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.7, min_child_weight=3,
        gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, eval_metric='logloss'
    )
    modelo_xgbr.fit(X_treino, y_treino)
    
    acc = accuracy_score(y_val, modelo_xgbr.predict(X_val))
    acuracias_xgbr.append(acc)
    previsoes_totais_xgbr += modelo_xgbr.predict_proba(X_teste)[:, 1] / skf.n_splits

print(f"Acurácia Média XGBoost Regularizado: {np.mean(acuracias_xgbr) * 100:.2f}%")
sub_xgbr = pd.DataFrame({'PassengerId': test_df['Id_Passageiro'], 'Survived': (previsoes_totais_xgbr >= 0.5).astype(int)})
sub_xgbr.to_csv('../dados/resultado_submission_xgboost_regularizado.csv', index=False)

Acurácia Média XGBoost Regularizado: 82.49%
